# Лабораторная работа № 4


## Сверточные нейронные сети



### Структура сверточной нейронной сети

#### Сверточный слой Conv2D
Существуют одномерные, двумерные и трехмерные сверточные слои.
В данной практической работе будут рассмотрены двумерные сверточные слои.

Модели сверточных нейронных сетей строятся на основе класса `Sequential`, для создания используется конструкция:

```python
model.add(Conv2D(32, (3, 3), padding='same', activation='relu', strides=(1,1)))
```

1. Первый параметр `32` - количество ядер (фильтров) свертки.
2. Второй параметр `(3, 3)` – размер ядра свертки.
3. Третий параметр `padding='same'` - тип заполнения краев, нужен для сохранения размеров изображения, по умолчанию значение `padding='valid'`.
4. Четвертый параметр `activation='relu'` – указание функции активации.
5. Пятый параметр `strides=(1, 1)` – необязательный, задает шаг смещения фильтра.

> [!note] Уточнение по параметрам
> В современных версиях TensorFlow/Keras аргумент `strides` по умолчанию равен `1` (или `(1, 1)` для 2D), поэтому его можно не указывать явно, если шаг равен 1. Также параметр `input_shape` указывается только в самом первом слое модели.


#### Различия полносвязного и сверточного слоев
Принципиальное различие между полносвязным и сверточным слоем состоит в том, что первый изучает глобальные характеристики изображения целиком, а второй - локальные шаблоны в небольших фрагментах изображения.


#### Принцип работы сверточного слоя
Основная задача сверточного слоя – выделить признаки во входном изображении ядрами свертки (фильтрами) и сформировать карту признаков в виде тензора.

**Карта признаков** или **карта активации** - это обработанное ядром свертки исходное изображение.

**Ядра свертки (фильтры)** - это тензоры одного размера (задается при настройке слоя). Количество ядер в слое определяет глубину выходного массива (т.е. количество ядер вместе с разрешением изображения определяют форму выходного тензора).

**Свёртка** – это операция вычисления нового значения на основе значения выбранного пикселя и значений окружающих его пикселей.

**Алгоритм свёртки:**
1. фильтр накладывается на левую верхнюю часть входящего изображения;
2. производится поэлементное умножение значений фильтра и значений пикселей изображения;
3. полученные значения складываются, сумма будет результатом свертки области наложения;
4. фильтр перемещается дальше по изображению (за смещение отвечает параметр `strides`, по умолчанию равен `(1,1)`, т.е. смещение на 1 пиксель вправо, при достижении конца строки сдвиг вниз на 1 пиксель, и вновь с начала строки);
5. в новом положении окна фильтра производится поэлементное умножение значений фильтра ...
6. повтор до тех пор, пока аналогичным образом не будут обработаны все участки.


#### Параметр Padding
При малом размере ядра фильтра, потеря в размерах выходного слоя существенна. При увеличении размера ядра размеры выходного слоя уменьшаются по отношению к оригинальному размеру входного изображения. Чтобы избежать потерь в разрешении выходных изображений, в настройке сверточного слоя используется параметр – заполнение (`padding`), позволяющий расширить полученное изображение по краям до того же размера, что и на входе свертки.

**Пример:** если картинка размером `5 x 5` пикселей, а ядро `3 x 3`, в итоге будет потеря в размерах.
**Решение:** использовать параметр `padding='same'`, тогда выходное изображение по внешней границе будет дополнено нулями (другие числа исказили бы информацию, а нули дают только изменение размера). Заполнение распределяется по границам.

> [!tip] Формула расчета выходного размера
> Общая формула для вычисления размера выходного признака (без учета автоматического padding='same'):
> $$ O = \lfloor \frac{I - K + 2P}{S} \rfloor + 1 $$
> Где:
> - $O$ — выходной размер
> - $I$ — входной размер
> - $K$ — размер ядра (kernel)
> - $P$ — размер padding (заполнения)
> - $S$ — шаг (stride)
>
> При `padding='same'` размер padding рассчитывается автоматически так, чтобы выходной размер совпадал с входным (при stride=1): $P = \lfloor \frac{K - 1}{2} \rfloor$.


#### Размеры ядра свертки
Слои в свёрточной нейронной сети используют ядра свертки в трех измерениях — ширина, высота, глубина (термин "глубина" относится к третьему измерению ядер, а не глубине самой нейронной сети, измеряемой в количестве слоёв).

Глубина фильтров первого слоя совпадает с количеством каналов входного изображения. Если на вход свёрточному слою подаётся RGB изображение (3 канала) и требуется получить 32 карты признаков, то свёрточный слой должен содержать в себе 32 фильтра глубиной 3.


#### Слой подвыборки (Pooling-слой)
Этот слой имеет ядро (окно фильтра).
Основное назначение - уменьшение размера изображения:
1. максимальное (`MaxPooling`),
2. среднее (`AveragePooling`)
3. суммарное (`SumPooling`) значение из окна фильтра.

Слой подвыборки имеет один обязательный параметр — `pool_size` (размер окна подвыборки), и один необязательный - `strides` (шаг смещения окна). Если `strides` не указан, то по умолчанию `strides = pool_size`, то есть окно смещается на размер фильтра.

Добавление слоя:
```python
model.add(MaxPooling2D(pool_size=(2, 2))) # где (2,2) – размер окна, в котором выбирается максимальное значение.
```


#### Финальная классификация данных
После прохождения через сверточные слои и слои подвыборки данные необходимо систематизировать и классифицировать. Для этого применяют полносвязные слои `Dense`.

Для преобразования сверточных слоев в слой `Dense` используют:
1. `Flatten()` - переводит многомерные входные данные в одномерный вектор, при этом размеры данных по всем осям перемножаются; дополнительных параметров нет. Например, входящее изображение формы `(28, 28, 3)` преобразуется в вектор формы `(2352)`.
2. `Reshape(...)` - смена формы данных. Требуется указать в скобках желаемую форму данных.

---



### Создание простой модели сверточной нейронной сети


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential()
input_shape = (img_height, img_width, channels)

# Первый сверточный слой
model.add(Conv2D(8, (3, 3), padding='same', activation='relu', input_shape=(28, 56, 3)))

Первый сверточный слой принимает на вход цветное изображение (3 канала) размерами 28 на 56 пикселей, форма входящего массива - `(28, 56, 3)`.

Внутри слоя к нему применяется свертка 8-ю фильтрами (3, 3) с шагом смещения (1, 1), а затем функция активации `relu`. `padding='same'`, `stride=(1,1)` по умолчанию;
вычислим `pad = (size - 1) / 2 = (3-1) /2 = 1`.

Используем формулу:
```text
output_h = (input_h + 2 * pad - size) // stride + 1 = (28 + 2 * 1 - 3) // 1 + 1 = 27 + 1 = 28
output_w = (input_w + 2 * pad - size) // stride + 1 = (56 + 2 * 1 - 3) // 1 + 1 = 55 + 1 = 56
```

In [ ]:
model.summary()

Добавим следующий сверточный слой c 5 фильтрами с ядром `(3, 2)`, шагом смещения `(2, 3)` и `padding='valid'`:

In [ ]:
# Второй сверточный слой
model.add(Conv2D(5, (3, 2), strides=(2, 3), padding='valid', activation='relu'))

Если `padding='valid'`, то по правилам `pad = 0`. Подставим значения:
```text
output_h = (input_h + 2 * pad - size) // stride + 1 = (28 + 2 * 0 - 3) // 2 + 1 = 25 // 2 + 1 = 12 + 1 = 13
output_w = (input_w + 2 * pad - size) // stride + 1 = (56 + 2 * 0 - 2) // 3 + 1 = 54 // 3 + 1 = 18 + 1 = 19
```
5 фильтров, форма данных на выходе - `(13, 19, 5)`.


In [ ]:
model.summary()

Применим слой `MaxPooling2D`:


In [ ]:
from tensorflow.keras.layers import MaxPooling2D

# Слой подвыборки
model.add(MaxPooling2D(pool_size=(3, 3)))

`MaxPooling2D` изменит форму данных следующим образом (учитывая, что `stride = pool_size`):
```text
output_h = (input - pool_size) // strides + 1 = (13 - 3) // 3 + 1 = 4
output_w = (input - pool_size) // strides + 1 = (19 - 3) // 3 + 1 = 6
```
Глубина в `MaxPooling2D` не меняется, выходная форма данных `(4, 6, 5)`.

In [ ]:
model.summary()

Далее слой `Flatten` преобразует входящий тензор в одномерный вектор.

In [ ]:
from tensorflow.keras.layers import Flatten

# Слой преобразования многомерных данных в одномерные
model.add(Flatten())

In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.layers import Dense

model.add(Dense(3, activation='softmax'))

---

### Распознавание изображений из набора данных MNIST

In [ ]:
from tensorflow.keras.datasets import mnist
# from tensorflow.keras.datasets import cifar10
# from tensorflow.keras.datasets import cifar100

from tensorflow.keras.models import Sequential
# Базовые слои для свёрточных сетей
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, Adadelta
from tensorflow.keras import utils
from tensorflow.keras.preprocessing import image
from google.colab import files

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import random
import math
import os
from google.colab import drive

%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### Подготовка данных

In [ ]:
# Загружаем MNIST
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train.shape

In [ ]:
# Выводим картинки по каждому классу
fig, axs = plt.subplots(1, 10, figsize=(25, 3))
for i in range(10):
    label_indexes = np.where(y_train == i)[0]
    index = random.choice(label_indexes)
    img = x_train[index]
    axs[i].imshow(Image.fromarray(img), cmap='gray')
plt.show()

In [ ]:
y_train = utils.to_categorical(y_train, 10)
y_test = utils.to_categorical(y_test, 10)

# Меняем формат данных MNIST
# Добавляем одну размерность для подачи на вход сверточного слоя
# чёрно-белые данные
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)

# Посмотрим форматы выборок перед обучением
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

#### Нейронная сеть
**Создаем архитектуру нейронной сети**

In [ ]:
# задаём batch_size
batch_size = 128

# Создаем последовательную модель
model = Sequential()

# Слой пакетной нормализации
model.add(BatchNormalization(input_shape=(28, 28, 1)))

# Первый сверточный слой
model.add(Conv2D(32, (3, 3), padding='same', activation='relu'))

# Второй сверточный слой
model.add(Conv2D(32, (3, 3), padding='same', activation='relu'))

# Первый слой подвыборки
model.add(MaxPooling2D(pool_size=(2, 2)))

# Слой регуляризации Dropout
model.add(Dropout(0.25))

model.add(Flatten())

# Полносвязный слой для классификации
model.add(Dense(256, activation='relu'))

# Слой регуляризации Dropout
model.add(Dropout(0.25))

# Выходной полносвязный слой
model.add(Dense(10, activation='softmax'))

# Компилируем сеть
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [ ]:
model.summary()

In [ ]:
# Обучаем сеть на данных mnist
history = model.fit(x_train,
                    y_train,
                    batch_size=batch_size,
                    epochs=15,
                    validation_data=(x_test, y_test),
                    verbose=1)